## Lesson Overview

**What this lesson teaches:** how to convert the chunks created in Lesson 11 into numerical embeddings with Voyage AI, then use those embeddings to retrieve sections related to a question.

**What's happening under the hood:**
1. Load the report and split it into Markdown sections.
2. Send all document chunks to an embedding model in one batch.
3. Embed a user's question with the same model.
4. Compare the query vector with every document vector using cosine similarity.
5. Rank the chunks and select the most relevant context.

The pattern to internalize: an embedding represents semantic meaning as a vector. Retrieval compares vectors; it does not ask the embedding model to write an answer.

# Lesson 12: Embeddings and Semantic Retrieval with Voyage AI

Lesson 11 produced chunks. This lesson turns those chunks into searchable vectors and retrieves the sections most relevant to a question. That gives us the **retrieval** in retrieval-augmented generation; answer generation will use the retrieved text in a later step.

## Where Embeddings Fit

```text
Document → chunks → document embeddings → vector index
                                              │
Question → query embedding ───────────────────┘
                                              ↓
                                    most similar chunks
```

Both sides must use the same embedding model. Voyage also distinguishes their roles with `input_type`: stored content uses `document`, while search text uses `query`.

## Setup

Install the packages once if your environment does not already have them:

```python
%pip install voyageai python-dotenv
```

The setup searches for `.env` in the current folder and in `Claude_API_Training`. It reads your existing `VOYAGE_API_KE` variable explicitly. The standard spelling, `VOYAGE_API_KEY`, is also accepted so you can rename it later without changing the notebook. The key is never printed.

In [1]:
import math
import os
import re
from pathlib import Path

try:
    import voyageai
except ImportError:
    voyageai = None

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(*args, **kwargs):
        return False

for env_path in (Path('.env'), Path('Claude_API_Training/.env')):
    if env_path.exists():
        load_dotenv(env_path, override=False)
        break

voyage_api_key = os.environ.get('VOYAGE_API_KE') or os.environ.get('VOYAGE_API_KEY')
client = voyageai.Client(api_key=voyage_api_key) if voyageai and voyage_api_key else None
embedding_model = 'voyage-3-large'

print(f'Embedding model: {embedding_model}')
print('Voyage client ready.' if client else 'Live embedding unavailable; check the package and API key.')

Embedding model: voyage-3-large
Voyage client ready.


## Load and Chunk the Report

We reuse Lesson 11's structure-aware strategy. The lookahead keeps each `##` heading attached to its section, giving every embedding a useful topic label. In a production pipeline, retain metadata such as document name, section heading, and chunk number alongside each vector.

In [2]:
def chunk_by_section(document_text):
    return [
        section.strip()
        for section in re.split(r'(?=^##\s)', document_text, flags=re.MULTILINE)
        if section.strip()
    ]

report_candidates = (Path('report.md'), Path('Claude_API_Training/report.md'))
report_path = next((path for path in report_candidates if path.exists()), None)
if report_path is None:
    raise FileNotFoundError('Could not find report.md.')

text = report_path.read_text(encoding='utf-8')
chunks = chunk_by_section(text)

print(f'Loaded {len(chunks)} chunks from {report_path}')
for index, chunk in enumerate(chunks):
    print(f'{index:>2}: {chunk.splitlines()[0]}')

Loaded 15 chunks from report.md
 0: # **Annual Interdisciplinary Research Review: Cross-Domain Insights**
 1: ## Executive Summary
 2: ## Table of Contents
 3: ## Methodology
 4: ## Section 1: Medical Research - Understanding XDR-471 Syndrome
 5: ## Section 2: Software Engineering - Project Phoenix Stability Enhancements
 6: ## Section 3: Financial Analysis - Q3 Performance and Outlook
 7: ## Section 4: Scientific Experimentation - Characterization of Material Composite XT-5
 8: ## Section 5: Legal Developments - Navigating IP Precedents and Regulatory Shifts
 9: ## Section 6: Product Engineering - Finalizing Model Zircon-5 Specifications
10: ## Section 7: Historical Research - Re-evaluating the Galveston Accords (1921)
11: ## Section 8: Project Management - Progress on Project Cerberus Phase 2B
12: ## Section 9: Pharmaceutical Development - Compound CTX-204b Phase IIa Update
13: ## Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011
14: ## Future Directions


## Generate Embeddings

The API accepts a list of texts, so the helper supports both one item and a batch. Batching document chunks is generally more efficient than making one request per chunk.

The result is a list of vectors: one vector for each input string, in the same order. A vector's individual numbers are not human-readable facts; meaning comes from comparing the complete vector with other vectors from the same model.

In [3]:
def generate_embeddings(texts, input_type, model=embedding_model):
    if client is None:
        raise RuntimeError('Install voyageai and configure VOYAGE_API_KE before requesting embeddings.')
    if input_type not in {'document', 'query'}:
        raise ValueError("input_type must be 'document' or 'query'")

    if isinstance(texts, str):
        texts = [texts]
    if not texts or any(not text.strip() for text in texts):
        raise ValueError('Provide at least one non-empty text')

    result = client.embed(texts, model=model, input_type=input_type)
    return result.embeddings

## Embed the Document Chunks

This is the indexing step. It makes a live Voyage API request and may incur a small charge. We use `document` because these vectors represent content that will be searched.

A real application normally computes document embeddings once and stores them. It should not re-embed an unchanged corpus for every question.

In [4]:
document_embeddings = None
if client is None:
    print('Skipping live embedding. Complete the setup, then rerun this cell.')
else:
    document_embeddings = generate_embeddings(chunks, input_type='document')
    print(f'Embedded {len(document_embeddings)} chunks')
    print(f'Each vector has {len(document_embeddings[0]):,} dimensions')
    print('First five values:', document_embeddings[0][:5])

Embedded 15 chunks
Each vector has 1,024 dimensions
First five values: [-0.06775207817554474, 0.01643814519047737, 0.014302929863333702, -0.015828924253582954, 0.007351024076342583]


## Cosine Similarity

Cosine similarity measures the angle between two vectors. Vectors pointing in a similar direction receive a higher score. This lets us rank sections by semantic similarity even when the query and document use different exact words.

```text
cosine similarity = dot(a, b) / (length(a) × length(b))
```

The helper checks vector dimensions and rejects zero-length vectors so errors fail clearly.

In [5]:
def cosine_similarity(vector_a, vector_b):
    if len(vector_a) != len(vector_b):
        raise ValueError('Vectors must have the same dimensions')

    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    magnitude_a = math.sqrt(sum(a * a for a in vector_a))
    magnitude_b = math.sqrt(sum(b * b for b in vector_b))
    if magnitude_a == 0 or magnitude_b == 0:
        raise ValueError('Cosine similarity is undefined for a zero vector')

    return dot_product / (magnitude_a * magnitude_b)

## Embed a Query and Retrieve the Top Chunks

Now we embed the question with `input_type='query'`. We compare that one vector with every document vector, sort by similarity, and return the highest-scoring chunks.

This notebook uses an in-memory Python list because the corpus is tiny. Larger systems use a vector database or a search engine with vector support.

In [7]:
def retrieve(query, chunks, document_embeddings, top_k=3):
    if not 1 <= top_k <= len(chunks):
        raise ValueError('top_k must be between 1 and the number of chunks')
    if len(chunks) != len(document_embeddings):
        raise ValueError('Each chunk must have exactly one embedding')

    query_embedding = generate_embeddings(query, input_type='query')[0]
    ranked = sorted(
        (
            {
                'index': index,
                'score': cosine_similarity(query_embedding, embedding),
                'text': chunk,
            }
            for index, (chunk, embedding) in enumerate(zip(chunks, document_embeddings))
        ),
        key=lambda item: item['score'],
        reverse=True,
    )
    return ranked[:top_k]

In [8]:
query = 'What happened during the cybersecurity incident and how was it contained?'

if document_embeddings is None:
    print('No document embeddings yet. Run the live embedding cell first.')
else:
    results = retrieve(query, chunks, document_embeddings, top_k=3)
    print('Query:', query)
    for rank, result in enumerate(results, start=1):
        heading = result['text'].splitlines()[0]
        print(f"{rank}. score={result['score']:.4f} | {heading}")

Query: What happened during the cybersecurity incident and how was it contained?
1. score=0.5502 | ## Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011
2. score=0.3583 | ## Section 2: Software Engineering - Project Phoenix Stability Enhancements
3. score=0.3513 | ## Table of Contents


## Inspect the Retrieved Context

Scores are useful for ranking, but always inspect the actual text. The highest-ranked chunk should contain evidence relevant to the question. Retrieval can still fail because of poor chunk boundaries, ambiguous wording, or content missing from the source.

In [9]:
if document_embeddings is None:
    print('No retrieval results to inspect yet.')
else:
    best_result = results[0]
    print(f"Similarity: {best_result['score']:.4f}\n")
    print(best_result['text'])

Similarity: 0.5502

## Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011

The Cybersecurity Operations Center successfully contained and remediated a targeted intrusion attempt tracked as `INC-2023-Q4-011`. Threat intelligence indicates the activity aligns with tactics, techniques, and procedures associated with the `ShadowNet Syndicate` threat actor group. Initial access was gained via a spear-phishing email targeting personnel within the finance department, potentially seeking data relevant to Section 3 (Financial Analysis). Endpoint detection and response (EDR) systems flagged anomalous process execution (`PID: 7812`) on workstation `WS-FIN-112`. Subsequent investigation identified malware (`SHA256:e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855`) attempting lateral movement towards server `SRV-FIN-03`. Containment involved isolating affected systems and blocking associated command-and-control infrastructure (IP `198.51.100.24`). Mitigat

## Local Sanity Checks

These checks do not call Voyage. They verify chunk structure and confirm the cosine-similarity calculation behaves as expected.

In [10]:
assert chunks
assert all(chunk.strip() for chunk in chunks)
assert any(chunk.startswith('## Section 10:') for chunk in chunks)
assert math.isclose(cosine_similarity([1, 0], [1, 0]), 1.0)
assert math.isclose(cosine_similarity([1, 0], [0, 1]), 0.0)
assert math.isclose(cosine_similarity([1, 0], [-1, 0]), -1.0)

print('All local checks passed.')

All local checks passed.


## Practice: Test Retrieval, Not Just the API

Try these one at a time and predict the top section before running:

1. **Exact identifier:** query for `ERR_MEM_ALLOC_FAIL_0x8007000E`. Does the software section rank first?
2. **Semantic wording:** ask `Which experimental material had problems under repeated stress?` without copying the report's wording.
3. **Ambiguous query:** search for `What happened to the incident?` Why might several sections compete? Rewrite it more precisely.
4. **Change top-k:** compare `top_k=1`, `3`, and `5`. When does extra context become noise?
5. **Chunking connection:** repeat the experiment with Lesson 11's character chunks. How do the results and readability change?

Reflection questions:
- Why must documents and queries use the same embedding model?
- Why do we use different `input_type` values?
- Which work happens once during indexing, and which work repeats for every query?
- Does a high similarity score prove that a chunk contains a correct answer?

## Summary

- Embeddings encode text as vectors for semantic comparison.
- Document chunks use `input_type='document'`; user searches use `input_type='query'`.
- All compared vectors must come from the same embedding model.
- Batch embedding is more efficient than one API request per chunk.
- Cosine similarity lets us rank chunks by their relationship to a query.
- Retrieval returns source context, not a generated answer.
- Chunk quality, metadata, and query wording all affect retrieval quality.

We now have the first half of a RAG pipeline: chunking, embedding, and retrieval. The next step is to place the retrieved chunks into a model prompt and require the answer to stay grounded in that context.